In [1]:
import pandas as pd
import openpyxl
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import re
from operator import itemgetter

In [19]:
pd.set_option('display.max_rows', None)

In [17]:
path_data = '/Users/emma/projects/cognoscolab/rassereno/data'

## Data extraction

In [18]:
raw_data_2025 = pd.read_csv(f"{path_data}/data_2025.csv")

In [4]:
# remove all the columns that are not needed
data_2025 = raw_data_2025.drop(columns=["EndDate", "Status", "IPAddress", "RecordedDate", "ResponseId", "RecipientLastName", "RecipientFirstName", "RecipientEmail", "ExternalReference", "LocationLatitude", "LocationLongitude", "DistributionChannel", "UserLanguage"])

# remove the first two rows which are 'index' of the columns
data_2025 = data_2025[2:]

# convert the progress values into integers
progress = data_2025['Progress'].values.astype(int)

# keep only the participations with more than 65% progress
data_2025 = data_2025[progress > 65].reset_index(drop=True)

## Participant ID creation

In [5]:
# ensure 'StartDate' is in datetime format
data_2025['StartDate'] = pd.to_datetime(data_2025['StartDate'])

# function to create participant IDs based on question 20 ('where do you study?')
def participant_id_question(q20_value, prefix):
    mask = data_2025["Q20"] == q20_value
    data_2025.loc[mask, 'participant_id'] = [f"{prefix}_{i+1:03d}" for i in range(mask.sum())]

# function to create participant IDs based only on specific date
def participant_id_date(dates, prefix):
    # convert single date string to list
    if isinstance(dates, str):
        dates = [dates]  

    # create a combined mask for all dates
    combined_mask = False
    for date in dates:
        mask = data_2025['StartDate'].dt.date == pd.to_datetime(date).date()
        combined_mask |= mask  # Combine masks with OR

    data_2025.loc[combined_mask, 'participant_id'] = [f"{prefix}_{i+1:03d}" for i in range(combined_mask.sum())]

# function to create participant IDs based on specific time intervals
def participant_id_time(start_time, end_time, prefix, start_idx=1):
    mask = (data_2025['StartDate'] >= start_time) & (data_2025['StartDate'] < end_time) & (data_2025["Q20"] == '3')
    count = mask.sum()
    data_2025.loc[mask, 'participant_id'] = [
        f"{prefix}_{i:03d}" for i in range(start_idx, start_idx + count)
    ]
    return mask, start_idx + count

In [6]:
participant_id_question('1', 'fisica')
participant_id_question('2', 'psicologia')
participant_id_question('3', 'liceo')

participant_id_date("2025-03-26", 'interfacce')
# fisica_stem is for participants that particiapated in the gruppo di ascolto per l'ansia da STEM
participant_id_date(["2025-05-21", "2025-05-29", "2025-06-17"], 'fisica_stem')

idx_sci = 1

mask_sci, idx_sci = participant_id_time("2025-05-13 11:40:00", "2025-05-13 11:45:00", 'liceo_sci', idx_sci)
mask_lin, _ = participant_id_time("2025-05-13 12:27:00", "2025-05-13 12:40:00", 'liceo_lin', 1)
mask_sci_2, idx_sci = participant_id_time("2025-05-30 07:50:00", "2025-05-30 08:55:00", 'liceo_sci', idx_sci+1)

# create participant_id for participant in row 65 (from liceo but some days later)
data_2025.loc[65, 'participant_id'] = f"liceo_sci_{mask_sci.sum()+1:03d}"
data_2025['participant_id']

0      interfacce_001
1      interfacce_002
2      interfacce_003
3      interfacce_004
4      interfacce_005
           ...       
94      liceo_sci_040
95      liceo_sci_041
96      liceo_sci_042
97      liceo_sci_043
98    fisica_stem_003
Name: participant_id, Length: 99, dtype: object

## Associations task

In [7]:
# these are the fileds used in the loops in qualtrics
field_1 = ["Scienza", "Equazione", "Matematica", "Programmazione", "Numeri", "Problema", "Dati"]
field_2 = ["Statistica", "Fisica", "Analisi", "Laboratorio", "Logica", "Informatica", "Esperimento"]
field_3 = ["Ipotesi", "Innovazione", "Scoperta", "Conoscenza", "Ricerca", "Sfida", "Stress"]
# Counselling é stata trasformata in Intellingenza_Artificiale dopo i primi 23 partecipanti
field_4 = ["Ansia", "Panico", "Scienziata", "Fallimento", "Imbarazzo", "Paura", "Intelligenza_Artificiale"] 
field_5 = ["Benessere", "Meditazione", "Docente", "Voto", "Compagni", "Genitori", "Regole"]
field_6 = ["Arte", "Vita", "Creatività", "Futuro", "Scuola", "Esame", "Scienziato"]

Dictonary to change keys 

{j}_QID71_{i}_{a} 

j = number of loop (7) 

i = $field (6) 

a = assosiation number (3) 

In [8]:
# make lists for the questions ID of the associations task questions and their text labels
keys_associations = []

keys_labels = []

# extract the questions ID and labels from the fields defined above
for j in range(1, 8):
    for i in range(1, 7):
        for a in range (1, 4):
            keys_associations.append(f"{j}_QID71_{i}_{a}") 
            keys_labels.append(globals()[f"field_{i}"][j - 1])

# create a dictionary to store the associations for each keyword
keywords_dict = defaultdict(list)

for key, value in data_2025.items():
    # 126 = 42 keywords * 3 associations
    for i in range(126):
        if key == keys_associations[i]: 
            if keys_labels[i] not in keywords_dict:
                keywords_dict[keys_labels[i]] = [] 
            keywords_dict[keys_labels[i]].append(value.tolist())

series_associations_df = pd.DataFrame.from_dict(keywords_dict)

# order of the cue words that will be presented in valence task
valence_keywords_order = ['Scienza', 'Statistica', 'Ipotesi', 'Ansia', 'Benessere', 'Equazione', 'Fisica', 'Innovazione', 'Panico', 'Meditazione', 'Matematica', 'Analisi', 'Scoperta', 'Scienziata', 'Docente', 'Programmazione', 'Laboratorio', 'Conoscenza', 'Fallimento', 'Voto', 'Numeri', 'Logica', 'Ricerca', 'Imbarazzo', 'Compagni', 'Problema', 'Informatica', 'Sfida', 'Paura', 'Genitori', 'Dati', 'Esperimento', 'Stress', "Intelligenza_Artificiale", 'Regole', 'Scienziato', 'Scuola', 'Creatività', 'Arte', 'Esame', 'Futuro', 'Vita']

# reorder the rows of df_keywords according to valence_keywords_order
series_associations_df = series_associations_df[valence_keywords_order]

# transpose the DataFrame to have cue words as rows and associations as columns
df_transposed = series_associations_df.T.reset_index()

df_transposed.columns = ['cue_word', 'association_1', 'association_2', 'association_3']

In [9]:
# set as index the cue words
associations_task_df = df_transposed.set_index('cue_word')

# explode the associations into separate rows and reset the index
associations_task_df = associations_task_df.apply(pd.Series.explode).reset_index()

# add the participant_id column multiplying the participant IDs for each cue word
associations_task_df["participant_id"] = np.tile(data_2025["participant_id"].values, len(valence_keywords_order))

associations_task_df = associations_task_df[['participant_id', 'cue_word', 'association_1', 'association_2', 'association_3']]

In [10]:
# Counselling é stata trasformata in Intellingenza_Artificiale dopo i primi 23 partecipanti
mask = associations_task_df['participant_id'].astype(str).str.startswith('interfacce') & (associations_task_df['cue_word'] == 'Intelligenza_Artificiale')

associations_task_df.loc[mask, 'cue_word'] = 'Counselling'

# associations_task_df.to_csv("extracted_data_2025/associations_task.csv", index=False)

In [11]:
# to check how nany cue word is repeated to see if there is one for each participant
word_counts = associations_task_df['cue_word'].value_counts().reset_index()

# we have 42 word * 97 participations = 4074

## Valence Task

### Structure of questions indices
j (9) = number of iteration (9 * 5(n questions) - 3 iterations)  

i (4) = number of text box (1: keyword; 2: first ass; 3: second ass; 4: third ass) 

1. first question 
{j}_Q20#{i}_1
2. second question 
{j}_QID63#{i}_1
3. third question 
{j}_QID64#{i}_1
4. fourth question 
{j}_Q15#{i}_1
5. fifth question 
{j}_Q16#{i}_1

### Extraction of association keys and values in valence task

In [12]:
# IDs of the questions of the valence task
q_ids = ["Q20", "QID63", "QID64", "Q15", "Q16"]

# creation of list to store the keys and values of the valence task
keys_valence= []
values_valence = []

# extraction of all the questions IDs (keys) and the values based on the q_id and their indices (see expalanation above)
for key, value in data_2025.items():
    for j in range(1, 10):
        for q_id in q_ids:
          for i in range(5):
            if key == f"{j}_{q_id}#{i}_1":
              # check if the column is not empty (if there are all NaN values)
              if not data_2025[key].iloc[:].isna().all():
                  keys_valence.append(key)
                  values_valence.append(data_2025[key])

# create a dictionary with the keys (q_ids) and values extracted from the valence task
valence_dict = dict(zip(keys_valence, values_valence))

### Combination of dataframes of valence and association tasks

In [13]:
# create lists for questions IDs labels
labels = []

# loop for the 4 associations valence scores (valence of the cue word and of the 3 associations) and make a single dictionary

    # take each of the cue word from valence_keywords_order and create labels for each association in the valence task
for word_key in valence_keywords_order:
    for i in range(4):
        labels.append(f"{word_key}_association_{i}")

# create a dict with the list of all questions IDs (previously created) and the label associated to each of them
id_to_label = dict(zip(keys_valence, labels))

In [14]:
# assign to the question IDs the question label so that i understand to what the scores are associeted
labeled_scores = pd.DataFrame.from_dict(valence_dict, orient='index').T
labeled_scores.rename(columns=id_to_label, inplace=True)

# melt to long format: each row is one valence_key per participant
melted_valence_df = labeled_scores.melt(var_name="key", value_name="valence")

# # assign an index to the df, based on the number of association (0 to 4) so that i can make separate columns for each of them
melted_valence_df["index"] = melted_valence_df["key"].str.extract(r'association_(\d)').astype(int)

# adds a new column ("row") to valence_df that tells the position of each row within its group based on the "index" column
melted_valence_df["row"] = melted_valence_df.groupby("index").cumcount()

# reshapes to wide format using as index the row as columns headers the index and as values the key and valence
df_pivoted = melted_valence_df.pivot(index="row", columns="index", values=["key", "valence"])

# create names for the columns based on their number of association and values 
df_pivoted.columns = [f"{col}_association_{i}" for col, i in df_pivoted.columns]

# reset the index without reassing it to a new variable
df_pivoted.reset_index(drop=True, inplace=True)

In [16]:
# concatenate the df with the labels for the valence task and the df of the association taks (previously created)
test_associations_valence = pd.concat([associations_task_df, df_pivoted], axis=1)

columns_order = [
    'participant_id', 
    'key_association_0',
    'cue_word',
    'valence_association_0',
    'key_association_1',
    'association_1',
    'valence_association_1',
    'key_association_2',
    'association_2',
    'valence_association_2',
    'key_association_3',
    'association_3',
    'valence_association_3'
]

# redorder the df based on columns_order
test_associations_valence = test_associations_valence[columns_order]

### Valence df creation and exporting to csv

In [20]:
path_out = '/Users/emma/projects/cognoscolab/rassereno/out/formatting/all_students_2025'

In [21]:
# clean the df to keep only the columns needed for the csv of the valence ans associations combined, making a copy first
valence_task_df = test_associations_valence.copy()

# remove the columns contaning the labels of the questions IDs in the valence task
valence_task_df = valence_task_df.drop(columns=['key_association_0', 'key_association_1', 'key_association_2', 'key_association_3'])

# rename the column related to the cue word in the valence task
valence_task_df = valence_task_df.rename(columns={'valence_association_0': 'valence_cue_word'})

# export the df to csv
valence_task_df.to_csv(f'{path_out}/valence_task.csv', index = False)

### Check of NaN values

In [22]:
# check which rows hav all NaN values in the valence task columns
nan_rows = valence_task_df[valence_task_df[['valence_cue_word', 'valence_association_1', 'valence_association_2', 'valence_association_3']].isna().all(axis=1)].index.tolist()

print(len(nan_rows))
print(nan_rows)

267
[23, 24, 26, 59, 60, 69, 122, 123, 125, 158, 159, 168, 221, 222, 224, 257, 258, 267, 320, 321, 323, 356, 357, 366, 419, 420, 422, 455, 456, 465, 518, 521, 551, 554, 557, 564, 566, 575, 617, 620, 650, 653, 656, 663, 665, 674, 716, 719, 749, 752, 755, 762, 764, 815, 818, 848, 851, 854, 861, 863, 914, 917, 947, 950, 953, 960, 962, 1013, 1049, 1052, 1059, 1112, 1148, 1151, 1158, 1211, 1247, 1250, 1257, 1310, 1346, 1349, 1356, 1409, 1445, 1448, 1455, 1508, 1544, 1545, 1554, 1556, 1607, 1643, 1644, 1653, 1655, 1706, 1742, 1743, 1752, 1754, 1786, 1805, 1806, 1841, 1842, 1851, 1853, 1904, 1905, 1940, 1941, 1950, 1952, 2003, 2039, 2040, 2042, 2049, 2051, 2102, 2138, 2139, 2141, 2148, 2150, 2201, 2237, 2238, 2240, 2247, 2249, 2300, 2336, 2337, 2339, 2346, 2348, 2382, 2399, 2435, 2436, 2438, 2445, 2447, 2454, 2498, 2499, 2501, 2534, 2537, 2597, 2598, 2600, 2633, 2636, 2696, 2697, 2699, 2732, 2735, 2795, 2796, 2798, 2831, 2834, 2894, 2895, 2897, 2930, 2933, 2993, 2994, 2996, 3026, 3029, 3030, 

## Parte Questionari

### Gender

In [23]:
# extracts gender information
gender = pd.DataFrame(data_2025['Q17']).replace({
    '1': 'M', # male
    '2': 'F', # female
    '4': 'NB', # non-binary
    '5': 'NA' # prefer not to say
})

gender.columns = ['gender']

### Competenze matematica

In [24]:
# math abilities self perception
percieved_abilities = pd.DataFrame(data_2025['Q16'])
percieved_abilities.columns = ['percieved_abilities']

# math grade
math_grade_uni = data_2025['Q20_1'].replace(31, '30L')
math_grade_liceo = data_2025['Q21_1'].replace(11,'10L')

# combine the two math grade columns, prioritizing university grades over liceo grades 
math_grade = math_grade_uni.combine_first(math_grade_liceo)
math_grade_df = pd.DataFrame({'math_grade': math_grade})

### Tutti i questionari

In [25]:
# Function to extract the data of some columns corresponding to a specific administrated questionnaire
# It also renames the columns with the name of the questionnaire and number of question

def extract_block(data, question_id, start=1, end=13, col_label = 'questionnaire'):
    list_questions = []
    for i in range(start, end):
        list_questions.append(data[f'{question_id}_{i}'])

    df =  pd.DataFrame(list_questions).transpose()
    df.columns = [f'{col_label}_{i}' for i in range(start, end)]

    return df

In [26]:
# Cognitive Flexibility Scale
cfs_data = extract_block(data_2025, 'Q22#1', 1, 13, col_label = 'cfs')

# STAI: State Anxiety
stai_data = extract_block(data_2025, 'QID77', 1, 19, col_label = 'state_anxiety')

# Big 5: 10 items
big5_data = extract_block(data_2025, 'QID84', 1, 11, col_label = 'big_5')

# MAS_IT reduced
masit_data = extract_block(data_2025, 'Q21#1', 1, 19, col_label = 'mas_it')

### Saving all the dfs into a csv file

In [28]:
# combines all the df and the participant_id into one and saves it to csv file
combined_dfs = pd.concat([data_2025['participant_id'],gender, percieved_abilities, math_grade, cfs_data, stai_data, big5_data, masit_data], axis=1)
combined_dfs.to_csv(f'{path_out}/questionnaires.csv', index=False)

In [29]:
df_liceo = combined_dfs[combined_dfs['participant_id'].str.startswith('liceo')]

In [30]:
df_liceo[['participant_id', 'gender']]

,participant_id,gender
28,liceo_sci_001,F
29,liceo_sci_002,F
30,liceo_sci_003,F
31,liceo_sci_004,F
32,liceo_sci_005,F
...,...,...
93,liceo_sci_039,F
94,liceo_sci_040,F
95,liceo_sci_041,F
96,liceo_sci_042,M


## Free text

In [31]:
# extract the text of the free text question (Q18)
text = pd.DataFrame(data_2025['Q18'])
text.columns = ['text']

# convert to string in lower case
text['text'] = text['text'].str.lower()

# remove trailing punctuation/whitespace, then add a full stop
text['text'] = text['text'].str.rstrip(".!? \n\t") + "."

# add participant_id
text = pd.concat([data_2025['participant_id'], text], axis=1)

text.to_csv(f'{path_out}/free_text.csv', index=False)